# Phase A - `StudentAudio2Landmark` : Audio (80D mel) -> 68x2 = 136 Landmarks (2D, da can chinh tu the dau)

Bo khung nay huan luyen mot mo hinh **Student** sieu nhe (1D-CNN + FC, ~3.1M tham so)
thay the cho pipeline nang cua MakeItTalk goc (Transformer / BiLSTM / LSTM-GAN).

## Pipeline

```
Audio .wav  --(AutoVC mel extractor)-->  Mel 80D  --(Student 1D-CNN + FC)-->  204D Landmarks  -->  Render 2D
```

| Giai doan | Noi dung | Trang thai |
| :--- | :--- | :--- |
| **Phase 1 (notebook nay)** | Hoc landmark chinh xac tu audio goc. `emotion_dim = 0`. | Chay duoc |
| **Phase 2 (tiep theo)** | Tai checkpoint Phase 1 len, gan them nhanh **Emotion Vector (alpha)`, fine-tune. | Hook da co san, cho du lieu MEAD |

## Diem quan trong ve kien truc

* Nhanh emotion duoc noi **sau** khoi convolution va **sau** buoc upsample thoi gian,
  tuc la ngay tai bien cua FC head -> toan bo kernel conv giu nguyen shape o Phase 1,
  chi co `fc[0]` no ra. Khong phai train lai tu dau o Phase 2.
* `emotion_gate` khoi tao bang `1`, cac cot moi cua `fc[0]` khoi tao bang `0`
  => **ngay sau khi nang cap, output giong het Phase 1 tung bit**, nhung gradient
  da chay duoc vao nhanh emotion ngay backward dau tien. (Neu zero-init ca hai dau
  thi tich bang 0 -> nhanh emotion mai mai khong hoc duoc.)

## Thu tu chay cac cell

| Cell | Noi dung | Colab | Local smoke |
| :--- | :--- | :---: | :---: |
| 1 | Moi truong, thu vien, `CFG` | bat buoc | bat buoc |
| 2 | Mount Drive (Colab) / kiem tra `G:\` (local) | bat buoc | bat buoc |
| 3 | Do chieu GitHub (chi Colab) + import model | bat buoc | bat buoc |
| 4 | Do chieu du lieu tren Drive | bat buoc | bat buoc |
| 5 | Index 6 file pickle, kiem tra ai <-> fl | bat buoc | bat buoc |
| 6 | `WindowDataset` / `FullSequenceDataset` | bat buoc | bat buoc |
| 7 | Khoi tao `StudentAudio2Landmark` + kiem tra kien truc | bat buoc | bat buoc |
| 8 | DataLoader | bat buoc | bat buoc |
| 9 | Vong lap huan luyen -> checkpoint len Drive | bat buoc | 1 epoch |
| 10 | Ve duong cong loss | nen chay | nen chay |
| 11 | **Evaluation** - nghiem thu LMD | bat buoc | bat buoc |
| 12 | Xuat `.npy` de render 2D | tuy chon | bo qua |

**Muon chi danh gia mot checkpoint co san (khong train lai):** chay **Cell 1 -> 2 -> 3 -> 4
-> 5 -> 6** roi nhay thang toi **Cell 11**. Cell 11 doc lap, tu tim checkpoint trong `OUT_DIR`
(xem muc "Cell nay doc lap" o duoi). Cell 7 va 8 khong can thiet cho Cell 11.

**Cell nao sinh file gi?**

| Cell | Ghi ra | Noi luu |
| :--- | :--- | :--- |
| 9 | `phaseA_student_last.pth`, `best.pth`, `epoch_NNN.pth`, `phaseA_history.json`, `phaseA_config.json`, `phaseA_loss_curves.png` | `OUT_DIR` tren Drive |
| 10 | `phaseA_loss_curves.png` (ghi de) | `OUT_DIR` |
| 11 | `phaseA_evaluation.json` | `OUT_DIR` |
| 12 | `pred_landmarks/*.npy` | `OUT_DIR` |

## Cach chay

1. **Google Colab (khuyen nghi - co GPU)**: `Runtime > Change runtime type > T4 GPU`,
   roi `Runtime > Run all`. Notebook tu mount `/content/drive`.
2. **Local (Windows)**: chay binh thuong trong VS Code. Notebook tu nhan dien
   `G:\\` va khong mount Drive. GPU local yeu -> nen chay o che do smoke test
   (`CFG['max_videos'] = 4`, `CFG['epochs'] = 1`) de kiem tra pipeline.
3. Checkpoint `.pth` duoc ghi dinh ky **truc tiep len Google Drive**, nen Colab bi
   ngat van resume duoc (`AUTO_RESUME = True`).

## Hop dong du lieu (da kiem chung bang probe)

| File | Kieu | Ghi chu |
| :--- | :--- | :--- |
| `autovc_retrain_mel_<split>_au.pickle` | `list[(mel float32 (F,80), (vid_id, 'xxxxx_ObamaNNNNN_audio.wav'))]` | Mel AutoVC |
| `autovc_retrain_mel_<split>_fl.pickle` | `list[(fl float64 (F,204), (vid_id, 'ObamaNNNNN.mp4', fps, dur, .., eyes, anchor, cams))]` | Landmark |
| `emb.pickle` | `dict[ytid] -> Tensor(256,)` | Speaker embedding (Phase 2, chua dung) |

* `<split>` in `{train, val, test}`.
* `au` va `fl` **duoc danh index song song**, so frame moi cap bang nhau (da verify).
* `fl` goc (204D, co z) duoc chuyen thanh `autovc_retrain_mel_<split>_fl2d_aligned.pickle` (136D) bang
  script `make_aligned.py`: (1) can chinh 3D tung frame ve mot tu the dau chuan bang cac diem cung
  (cau mui 27-30, 4 khoe mat 36/39/42/45), (2) bo truc z. Tham chieu tinh tu tap train (`align_ref.npz`).
  Khi do model chi phai hoc khau hinh, khong phai doan tu the dau. Chi can chuan hoa mel.
* Mel chuan hoa bang `MEAN_STD_AUTOVC_RETRAIN_MEL_AU.txt` (80 mean + 80 std).


## Cell 1 - Moi truong, thu vien va cau hinh


In [ ]:
import os, re, sys, gc, json, math, glob, time, random, pickle, shutil, platform
from pathlib import Path
from collections import OrderedDict

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError('Thieu thu vien (%s). Chay: pip install torch numpy matplotlib' % exc)

IN_COLAB = 'google.colab' in sys.modules

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_NAME = (torch.cuda.get_device_name(0) if DEVICE.type == 'cuda'
               else 'CPU (%s)' % platform.processor())

print('Python       :', sys.version.split()[0], '|', platform.system())
print('PyTorch      :', torch.__version__)
print('Moi truong   :', 'Google Colab' if IN_COLAB else 'Local')
print('Device       :', DEVICE, '|', DEVICE_NAME, end='')
if DEVICE.type == 'cuda':
    print(' | VRAM %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024 ** 3))
else:
    print(' | (khong co GPU - nen day notebook len Colab de train that)')

# ---------------------------------------------------------------- CAC DUONG DAN
REPO_URL = ''   # TODO: dien URL repo cua ban sau khi push len Git, vi du
                #       'https://github.com/<user>/MakeItTalk.git'

if IN_COLAB:
    DRIVE_MY = Path('/content/drive/MyDrive')
    REPO_DIR  = Path('/content/MakeItTalk')
    OUT_DIR   = Path('/content/drive/MyDrive/DAT301m/Data_student/student_a2l/phaseA')
else:
    DRIVE_MY = Path('G:/My Drive')
    REPO_DIR  = Path.cwd().resolve()
    OUT_DIR   = DRIVE_MY / 'DAT301m' / 'Data_student' / 'student_a2l' / 'phaseA'

# Cho huan luyen THUC SU tren Colab: giam so mau/epoch de 1 epoch chay vai phut,
# tranh bi ngat ket noi truoc khi kip luu checkpoint dau tien.
#   COLAB_FAST = True  ->  35k mau/epoch, 4k mau validate  (~2-3 phut/epoch tren T4)
#   COLAB_FAST = False -> 200k mau/epoch, 20k mau validate (~10-20 phut/epoch)
# Co the ghi de ma khong can sua file: os.environ['COLAB_FAST'] = '0'
COLAB_FAST_SRC = os.environ.get('COLAB_FAST', '1').strip().lower()
COLAB_FAST = COLAB_FAST_SRC not in ('0', 'false', 'no', 'off')

# --------------------------------------------------------------- HINH DANG LANDMARK (2D)
NUM_POINTS = 68
COORDS     = 2                      # 2D: (x, y); da can chinh tu the dau va bo z
LM_DIM     = NUM_POINTS * COORDS    # 136

# --------------------------------------------------------------- SIEU THAM SO
CFG = dict(
    # du lieu
    num_window_frames = 18,     # giong MakeItTalk goc
    num_window_step   = 1,
    use_std_normalized_fl = True,   # True -> doc fl bang float32
    fl_name           = 'fl2d_aligned',   # autovc_retrain_mel_<split>_<fl_name>.pickle (2D, da can chinh)
    # toi uu
    epochs            = 40,
    batch_size        = 256,
    # num_workers=0 la bat buoc tren Windows: Dataset khai bao trong notebook
    # khong pickle duoc -> DataLoader da tien trinh chet voi PicklingError.
    num_workers       = 0 if (os.name == 'nt' and not IN_COLAB) else 2,
    lr                = 3e-4,
    weight_decay      = 1e-5,
    warmup_epochs     = 2,
    grad_clip         = 1.0,
    amp               = True,
    # ham loss: D-LL + D-VL + D-A (dat 0.0 cho 2 so hang cuoi de co baseline thuan MSE)
    lambda_dll        = 1.0,
    lambda_dvl        = 0.1,
    lambda_dal        = 0.05,
    # gioi han so mau moi epoch (tap train co ~1.1M cua so -> lay mau ngau nhien)
    samples_per_epoch = 35_000 if COLAB_FAST else 200_000,
    val_samples       =  4_000 if COLAB_FAST else 20_000,
    # checkpoint
    ckpt_every_steps  = 2000,
    keep_last_n       = 3,
    # danh gia
    # Model CHI duoc train tren cua so `num_window_frames` frame. Suy luan tren mot doan
    # dai (VD 4000 frame) dua BatchNorm cua khoi conv ra ngoai phan phoi da hoc: khi do dai
    # chuoi sau downsample (= T/16) giam ve 2, lop conv thu hai trong _ConvBlock1d
    # (stride=1, k=3, pad=1) khong con truc thoi gian de tich chap -> dau ra gan nhu khong
    # con phu thuoc vao input. Vi vay mac dinh danh gia bang CUA SO TRUOT.
    # Do do: 'window' = che do dung | 'chunk' = chi de chan doan (se bao LMD rat xau).
    eval_mode          = 'window',
    eval_window_stride = 8,      # buoc truot khi eval_mode='window' (8 nhanh ~6x, chat luong nhu buoc 1)
    eval_chunk_frames  = 4000,   # chi dung khi eval_mode='chunk'
    eval_overlap       = 64,     # chi dung khi eval_mode='chunk'
    eval_accept_gain   = 25.0,   # nghiem thu: LMD Mouth phai tot hon baseline it nhat X %.
                                 # 25 chi la gia tri khoi dau, hay chinh lai sau lan chay dau tien.
    # Chi dinh tay checkpoint can danh gia. De trong '' thi Cell 11 tu tim theo thu tu:
    # CKPT_BEST/CKPT_LAST -> OUT_DIR/phaseA_student_best.pth -> last.pth -> epoch moi nhat.
    # Chi can dien khi ban muon nghiem thu mot file .pth nam o noi khac.
    eval_ckpt          = '',
    # debug: > 0 se chi lay `max_videos` video moi split (dung cho smoke test local)
    max_videos        = 0,
    # Neu tu dong do tim that bai (VD: thu muc Drive dat qua sau), dien duong dan
    # tuyet doi toi thu muc chua file .pickle. Vi du Colab:
    #   '/content/drive/MyDrive/DAT301m/Data_examble/Data_maketalk'
    # Vi du local: r'G:\.shortcut-targets-by-id\1EwuAy3...\data'
    data_root_override = '',
)
# A robust absolute path for the config file, so we keep it in the same place as the notebook if possible.
CONFIG_PATH = OUT_DIR / 'phaseA_config.json'

print('COLAB_FAST =', COLAB_FAST, '| nguon:', COLAB_FAST_SRC)
print('IN_COLAB   =', IN_COLAB)
print('DEVICE     =', DEVICE, '|', DEVICE_NAME)

for k, v in CFG.items():
    print('  %-22s %s' % (k, v))


## Cell 2 - Mount Google Drive (chi tren Colab) va nap thu vien thieu


In [ ]:
if IN_COLAB:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        print('Drive mounted ->', DRIVE_MY)
    except Exception as exc:
        raise RuntimeError('Khong mount duoc Google Drive: %s' % exc)
else:
    print('Local mode: bo qua mount. Kiem tra o G:\\ ...')
    print('  G:\\ ton tai      :', Path('G:/').exists())
    print('  My Drive ton tai :', DRIVE_MY.exists())


## Cell 3 - Do chieu sang GitHub (chi Colab): clone repo + dong bo source

Neu ban da push repo len Git thi chi can dien `REPO_URL` o Cell 1, notebook se tu clone.
Neu chua push: notebook se tim mot ban copy `src/models/model_audio2landmark.py` **co chua
class `StudentAudio2Landmark`** ngay tren Google Drive, hoac dung repo local hien tai.


In [ ]:
MODEL_REL = Path('src') / 'models' / 'model_audio2landmark.py'


def _repo_is_valid(root):
    f = Path(root) / MODEL_REL
    if not f.exists():
        return False
    try:
        return 'class StudentAudio2Landmark' in f.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return False


def _search_drive_for_repo(base, max_depth=5):
    """Tim thu muc goc repo tren Drive (co chua model_audio2landmark.py ban moi)."""
    base = Path(base)
    if not base.exists():
        return None
    base_depth = len(base.parts)
    for cur, dirs, files in os.walk(base):
        if len(Path(cur).parts) - base_depth > max_depth:
            dirs[:] = []
            continue
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        if 'model_audio2landmark.py' in files:
            root = Path(cur).parent.parent
            if _repo_is_valid(root):
                return root
    return None


def _search_upwards_for_repo(start, max_up=4):
    """Tu thu muc hien tai di nguoc len tim thu muc goc repo.

    Can thiet khi mo notebook bang VS Code / Jupyter: luc do cwd la thu muc
    `notebooks/`, khong phai goc repo.
    """
    cur = Path(start).resolve()
    for _ in range(max_up + 1):
        if _repo_is_valid(cur):
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return None


def ensure_repo():
    if _repo_is_valid(REPO_DIR):
        print('Dung repo tai:', REPO_DIR)
        return REPO_DIR
    if not IN_COLAB:
        found = _search_upwards_for_repo(Path.cwd()) or _search_upwards_for_repo(
            Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd())
        if found:
            print('Tim thay repo (di nguoc tu cwd):', found)
            return found
    if REPO_URL:
        import subprocess
        if Path(REPO_DIR).exists() and not (Path(REPO_DIR) / '.git').exists():
            shutil.rmtree(REPO_DIR, ignore_errors=True)
        if not (Path(REPO_DIR) / '.git').exists():
            print('Cloning', REPO_URL)
            subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
        else:
            subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
        if _repo_is_valid(REPO_DIR):
            print('Dung repo tai:', REPO_DIR)
            return REPO_DIR
        print('[canh bao] Repo clone ve nhung khong chua StudentAudio2Landmark.')
    if IN_COLAB:
        print('Dang tim ban copy repo tren Google Drive ...')
        found = _search_drive_for_repo(DRIVE_MY)
        if found:
            print('Tim thay:', found)
            return found
    raise FileNotFoundError(
        'Khong tim thay repo chua class StudentAudio2Landmark.\n'
        '  -> Local: mo notebook tu thu muc goc repo, hoac chay tu thu muc goc.\n'
        '  -> Colab cach 1 (khuyen nghi): dien REPO_URL o Cell 1 roi chay lai.\n'
        '  -> Colab cach 2: copy ca thu muc repo (co src/models/) len Google Drive.')

# Doi chieu: nap class tu source vua tim duoc
REPO_DIR = ensure_repo()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.models.model_audio2landmark import (          # noqa: E402
    StudentAudio2Landmark, STUDENT_MOUTH_IDX,
    STUDENT_AUDIO_DIM,
)

print('Import OK  |  audio_dim=%d  landmark_dim=%d  mouth_idx=%d diem'
      % (STUDENT_AUDIO_DIM, LM_DIM, len(STUDENT_MOUTH_IDX)))


## Cell 4 - Do chieu sang du lieu tren Google Drive

Tim thu muc chua `autovc_retrain_mel_train_au.pickle`. Notebook ho tro ca 2 truong hop:

* Thu muc **that su chua du lieu** (tren may local la
  `G:\\.shortcut-targets-by-id\\1EwuAy3j1b9Zc1MsidUfxG_pJGc_cV60O\\data`).
* Thu muc shortcut / Shared Drive tren Colab
  (`/content/drive/.shortcut-targets-by-id/*/data`, `/content/drive/Shareddrives/*/data`).


In [ ]:
AU_TRAIN = 'autovc_retrain_mel_train_au.pickle'


def _has_data(p):
    return (Path(p) / AU_TRAIN).exists()


def _walk_for_data(bases, max_depth=4):
    """Quet co gioi han do sau de tim thu muc chua file pickle cua tap train."""
    for base in bases:
        base = Path(base)
        if not base.exists():
            continue
        base_depth = len(base.parts)
        for cur, dirs, files in os.walk(base):
            if len(Path(cur).parts) - base_depth > max_depth:
                dirs[:] = []
                continue
            dirs[:] = [d for d in dirs if not d.startswith('.')]
            if AU_TRAIN in files:
                return Path(cur)
    return None


def find_data_root():
    # 0) Uu tien tuyet doi: nguoi dung dien tay o CFG.
    if CFG.get('data_root_override'):
        p = Path(os.path.expanduser(str(CFG['data_root_override'])))
        if _has_data(p):
            print('Dung data_root_override tu CFG.')
            return p
        raise FileNotFoundError(
            "CFG['data_root_override'] = %s nhung khong thay %s trong do." % (p, AU_TRAIN))

    # 1) Duong dan quen thuoc (nhanh, khong can quet cay).
    cands = [
        DRIVE_MY / 'DAT301m' / 'Data_student' / 'Data_maketalk_landmard_task_a',
        DRIVE_MY / 'DAT301m' / 'Data_examble' / 'Data_maketalk',
    ]
    for p in cands:
        if _has_data(p):
            return p

    # 2) Quet Google Drive: ca MyDrive thuong, Thu muc dung chung (shortcut) va
    #    Shared Drives. Do sau 4 la du cho `.../data/*.pickle`.
    if IN_COLAB:
        drive_subs = sorted(glob.glob('/content/drive/*/'))
        bases = [DRIVE_MY] + [Path(d) for d in drive_subs]
    else:
        bases = [DRIVE_MY, Path('G:/.shortcut-targets-by-id')]
    found = _walk_for_data(bases, max_depth=4)
    if found:
        return found

    raise FileNotFoundError(
        'Khong tim thay %s.\n'
        '  -> Cach chac chan nhat: dien duong dan thu muc chua file pickle vao\n'
        "     CFG['data_root_override'] o Cell 1 roi chay lai Cell 4.\n"
        '  Cac ung vien da thu truc tiep:\n    %s\n'
        '  Da quet cay (do sau 4) tu: %s'
        % (AU_TRAIN, '\n    '.join(str(c) for c in cands),
           ', '.join(str(b) for b in bases)))


DATA_ROOT = find_data_root()
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT :', DATA_ROOT)
print('OUT_DIR   :', OUT_DIR)

# --- File chuan hoa nam trong repo -------------------------------------------------
MEAN_STD_PATH = REPO_DIR / 'src' / 'dataset' / 'utils' / 'MEAN_STD_AUTOVC_RETRAIN_MEL_AU.txt'
for p in (MEAN_STD_PATH,):
    print('%-28s %s' % (p.name, 'OK' if p.exists() else 'THIEU!'))


## Cell 5 - Index du lieu: ghep cap `au` <-> `fl` va kiem tra tinh dong bo

Hai pickle duoc ghep theo **index**, nhung ten file khac nhau
(`00037_Obama00037_audio.wav` vs `Obama00037.mp4`), nen phai doi chieu bang **ID so**
chu khong phai bang chuoi tho.


In [ ]:
from torch.utils.data import Dataset, DataLoader


def _vid_key(name):
    m = re.search(r'(\d{4,6})', str(name))
    return m.group(1) if m else str(name)


def load_split(split, max_videos=0):
    """Doc cap (au, fl) cua mot split.

    Tra ve (au, fl, names): au/fl la 2 list ndarray float32 (F,80) va (F,204),
    names la ten file audio de dat ten file .npy khi export.
    """
    au_path = DATA_ROOT / ('autovc_retrain_mel_%s_au.pickle' % split)
    fl_path = DATA_ROOT / ('autovc_retrain_mel_%s_%s.pickle' % (split, CFG['fl_name']))
    for p in (au_path, fl_path):
        if not p.exists():
            raise FileNotFoundError('Thieu file: %s' % p)

    with open(au_path, 'rb') as f:
        au_raw = pickle.load(f)
    with open(fl_path, 'rb') as f:
        fl_raw = pickle.load(f)

    n = min(len(au_raw), len(fl_raw))
    if len(au_raw) != len(fl_raw):
        print('[canh bao] %s: au co %d, fl co %d -> dung %d cap dau'
              % (split, len(au_raw), len(fl_raw), n))

    # --- kiem tra ghep cap (chi vai mau dau, de khong ton thoi gian) ---------------
    for i in range(min(4, n)):
        ka, kf = _vid_key(au_raw[i][1][1]), _vid_key(fl_raw[i][1][1])
        tu, tf = int(au_raw[i][1][0]), int(fl_raw[i][1][0])
        assert tu == tf and ka == kf, (
            'Lech cap tai index %d: au=(%s,%s) fl=(%s,%s)' % (i, tu, ka, tf, kf))

    au = [np.asarray(d[0], dtype=np.float32) for d in au_raw[:n]]
    fl = [np.asarray(d[0], dtype=np.float32 if CFG['use_std_normalized_fl'] else np.float64)
          for d in fl_raw[:n]]
    names = [str(au_raw[i][1][1]) for i in range(n)]
    assert all(f.shape[1] == LM_DIM for f in fl), (
        'fl phai co %d cot (2D da can chinh), hien co %d. Dung dung file %s?'
        % (LM_DIM, fl[0].shape[1], fl_path.name))
    del au_raw, fl_raw
    gc.collect()

    n_frames_total = sum(a.shape[0] for a in au)
    n_bad = sum(1 for a, b in zip(au, fl) if a.shape[0] != b.shape[0])
    print('%-5s | %3d video | %9s frame | lech frame: %d'
          % (split, len(au), '{:,}'.format(n_frames_total), n_bad))
    assert n_bad == 0, 'Co %d video lech so frame giua au va fl' % n_bad

    if max_videos and len(au) > max_videos:
        au, fl, names = au[:max_videos], fl[:max_videos], names[:max_videos]
        print('        -> cat con %d video (che do debug)' % max_videos)

    return au, fl, names


MAXV = CFG['max_videos']
AU, FL, NAMES = {}, {}, {}
for sp in ('train', 'val', 'test'):
    AU[sp], FL[sp], NAMES[sp] = load_split(sp, MAXV)

# --- Nap thong ke chuan hoa --------------------------------------------------------
mel_ms = np.loadtxt(MEAN_STD_PATH, dtype=np.float64).reshape(2, STUDENT_AUDIO_DIM)
MEL_MEAN = mel_ms[0].astype(np.float32)
MEL_STD  = np.maximum(mel_ms[1].astype(np.float32), 1e-5)
FL_MEAN  = np.concatenate(FL['train'], axis=0).mean(axis=0).astype(np.float32)   # (136,) dung de retarget

mel_all = np.concatenate(AU['train'], axis=0)
print()
print('Mel   | mean %.4f  std %.4f  min %.4f  max %.4f' %
      (mel_all.mean(), mel_all.std(), mel_all.min(), mel_all.max()))
del mel_all
print('Norm  | mel_mean[0:3]=%s  mel_std[0:3]=%s' %
      (np.round(MEL_MEAN[:3], 4), np.round(MEL_STD[:3], 4)))
print('FL    | min %.4f  max %.4f  |  2D da can chinh tu the dau, dim=%d'
      % (FL['train'][0].min(), FL['train'][0].max(), FL['train'][0].shape[1]))


## Cell 6 - Dataset

* **Train/Val**: cat cua so truot `num_window_frames=18`, `num_window_step=1` (giong ban goc),
  nhung chi luu **chi so cua so** vao mot mang `int64` -> khong ton RAM.
* **Test (eval)**: tra ve **ca chuoi** cua tung video, vi `D-VL` / `D-A` can tinh dao ham
  theo thoi gian lien tuc, neu cat cua so se bi sai o bien.


In [ ]:
class WindowDataset(Dataset):
    """Cua so truot (video_idx, start_frame) -> (mel_norm (18,80), fl (18,204))."""

    def __init__(self, au, fl, mean, std, window=18, step=1, min_frames=None):
        self.au, self.fl = au, fl
        self.mean, self.std = mean, std
        self.window, self.step = int(window), int(step)
        min_frames = self.window if min_frames is None else int(min_frames)
        idx = [(v, s) for v, a in enumerate(au)
               if a.shape[0] >= min_frames
               for s in range(0, a.shape[0] - self.window + 1, self.step)]
        self.index = np.asarray(idx, dtype=np.int64)
        del idx

    def __len__(self):
        return self.index.shape[0]

    def __getitem__(self, i):
        v, s = self.index[i]
        e = s + self.window
        mel = self.au[v][s:e]
        fl = self.fl[v][s:e]
        mel = (mel - self.mean) / self.std
        return torch.from_numpy(np.ascontiguousarray(mel)), \
               torch.from_numpy(np.ascontiguousarray(fl))

    def subset(self, n, seed=SEED):
        """Sampler tra ve n chi so ngau nhien (doi moi epoch)."""
        return SubsetSampler(len(self), n, seed)


class FullSequenceDataset(Dataset):
    """Tra ve nguyen video: (mel_norm (T,80), fl (T,204), vid_khong_chuan_hoa)."""

    def __init__(self, au, fl, mean, std, window=18, name=''):
        self.au, self.fl = au, fl
        self.mean, self.std = mean, std
        self.window = int(window)
        self.name = name
        self.keep = [i for i, a in enumerate(au) if a.shape[0] >= self.window]

    def __len__(self):
        return len(self.keep)

    def __getitem__(self, i):
        v = self.keep[i]
        mel = (self.au[v] - self.mean) / self.std
        return torch.from_numpy(np.ascontiguousarray(mel)), \
               torch.from_numpy(np.ascontiguousarray(self.fl[v])), v


class SubsetSampler(torch.utils.data.Sampler):
    """Moi epoch boc ngau nhien `n` chi so trong tap train."""

    def __init__(self, total, n, seed=SEED):
        self.total = int(total)
        self.n = min(int(n), self.total)
        self.epoch = 0
        self.seed = int(seed)

    def __iter__(self):
        g = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        return iter(g.choice(self.total, size=self.n, replace=False).tolist())

    def __len__(self):
        return self.n


WIN = CFG['num_window_frames']
train_ds = WindowDataset(AU['train'], FL['train'], MEL_MEAN, MEL_STD, WIN, CFG['num_window_step'])
val_ds   = WindowDataset(AU['val'],   FL['val'],   MEL_MEAN, MEL_STD, WIN, CFG['num_window_step'])
test_ds  = FullSequenceDataset(AU['test'], FL['test'], MEL_MEAN, MEL_STD, WIN, name='test')

x, y = train_ds[0]
print('WindowDataset       : %10s cua so | x=%s y=%s' % ('{:,}'.format(len(train_ds)), tuple(x.shape), tuple(y.shape)))
print('WindowDataset (val) : %10s cua so' % '{:,}'.format(len(val_ds)))
print('FullSequenceDataset : %10s video  | T = %s' % (len(test_ds), [test_ds[i][0].shape[0] for i in range(len(test_ds))][:5], ))


## Cell 7 - Khoi tao `StudentAudio2Landmark` va kiem tra kien truc

Kiem tra nhanh 3 dieu kien truoc khi train:

1. Forward `(B, T, 80) -> (B, T, 204)` voi do dai T bat ky (ke ca T le).
2. So tham so nam trong ngan sach "sieu nhe".
3. Nang cap Phase 2 (`set_emotion_dim`) phai **khong lam thay doi output Phase 1**.


In [ ]:
def build_model(emotion_dim=0, **overrides):
    """Tao model. `emotion_dim=0` cho Phase 1, `>0` cho Phase 2."""
    kwargs = dict(
        audio_dim=STUDENT_AUDIO_DIM,
        landmark_dim=LM_DIM,
        emotion_dim=emotion_dim,
        cnn_channels=(128, 256, 256, 384),
        kernel_size=5,
        strides=(2, 2, 2, 2),
        dilations=(1, 1, 1, 1),
        refine_dilation=2,
        fc_hidden=512,
        drop_out=0.2,
        upsample='interp',      # khong co tham so -> tot cho edge device
        emotion_fuse='concat',
    )
    kwargs.update(overrides)
    return StudentAudio2Landmark(**kwargs)


def smoke_test_model():
    m = build_model().to(DEVICE)
    n = m.num_parameters()
    assert n < 5_000_000, 'Model qua nang: %d tham so' % n
    with torch.no_grad():
        for T in (18, 137, 500):
            out = m(torch.randn(2, T, STUDENT_AUDIO_DIM, device=DEVICE))
            assert tuple(out.shape) == (2, T, LM_DIM), out.shape
        # layout (B, F, T) cung phai chay
        assert tuple(m(torch.randn(1, STUDENT_AUDIO_DIM, 137, device=DEVICE),
                       input_layout='bft').shape) == (1, 137, LM_DIM)

    # --- kiem tra duong nang cap Phase 2 ------------------------------------------
    m = m.cpu().eval()
    ref = torch.randn(2, 64, STUDENT_AUDIO_DIM)
    with torch.no_grad():
        before = m(ref)
        m2 = build_model(emotion_dim=8)
        m2.load_state_dict_flexible(m.state_dict())
        m2.eval()
        after = m2(ref)
        assert torch.allclose(before, after, atol=1e-6), \
            'Nang cap Phase 2 lam thay doi output Phase 1!'
    print('Kien truc OK  |  %.3f M tham so  |  downsample x%d  |  Phase-2 upgrade la no-op'
          % (n / 1e6, m.downsample_factor))
    return m


_MODEL = smoke_test_model()
_MODEL.cpu()
print(json.dumps(_MODEL.config(), indent=2, ensure_ascii=False))
del _MODEL
gc.collect()
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None


## Cell 8 - DataLoader


In [ ]:
pin = DEVICE.type == 'cuda'
train_loader = DataLoader(
    train_ds, batch_size=CFG['batch_size'], sampler=train_ds.subset(CFG['samples_per_epoch']),
    num_workers=CFG['num_workers'], pin_memory=pin, drop_last=True,
    persistent_workers=CFG['num_workers'] > 0,
)
# Tap val CO DINH: cung cac cua so o moi epoch, nen val_LMD giua cac epoch so sanh duoc
# va viec chon best.pth theo val_LMD khong bi nhieu do doi mau.
_val_idx = np.sort(np.random.default_rng(SEED).choice(
    len(val_ds), size=min(CFG['val_samples'], len(val_ds)), replace=False))
val_loader = DataLoader(
    torch.utils.data.Subset(val_ds, _val_idx.tolist()), batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'], pin_memory=pin, drop_last=False,
    persistent_workers=CFG['num_workers'] > 0,
)
print('Train: %d mau/epoch | Val: %d mau/epoch | batch=%d'
      % (len(train_loader.sampler), len(val_loader.sampler), CFG['batch_size']))


## Cell 9 - Vong lap huan luyen (AMP + cosine warmup + checkpoint len Drive)

Loss = `lambda_dll * D-LL + lambda_dvl * D-VL + lambda_dal * D-A`

* `D-LL` (MSE vi tri) la so hang chinh.
* `D-VL` (MSE van toc, dao ham bac 1) va `D-A` (MSE gia toc, dao ham bac 2) giup mo hinh
  khong bi rung/giat giua cac frame.
* Checkpoint ghi len Drive moi `ckpt_every_steps` step + cuoi moi epoch.
* `AUTO_RESUME = True`: Colab bi ngat -> chay lai cell nay se tu tiep tuc tu `last.pth`.

**`best.pth` duoc chon theo val LMD (thuoc do nghiem thu), khong theo val loss.** Ly do:
sau diem toi uu, val loss (bi chi phoi boi MSE vi tri) va val LMD khong con di cung chieu -
thuc nghiem cho thay val LMD dat cuc tieu som roi tang dan trong khi val loss van giam.
Log moi epoch in ca `val_LMD` de theo doi truc tiep.


In [ ]:
AUTO_RESUME  = True
CKPT_LAST    = OUT_DIR / 'phaseA_student_last.pth'
CKPT_BEST    = OUT_DIR / 'phaseA_student_best.pth'
HISTORY_PATH = OUT_DIR / 'phaseA_history.json'


def compute_loss(pred, gt, cfg):
    """pred/gt: (B, T, 204). Tra ve (tong, d_ll, d_vl, d_a) dang tensor."""
    d_ll = F.mse_loss(pred, gt)
    d_vl = pred.new_zeros(())
    d_a  = pred.new_zeros(())
    if pred.shape[1] > 2:
        pv, gv = pred[:, 1:] - pred[:, :-1], gt[:, 1:] - gt[:, :-1]
        d_vl = F.mse_loss(pv, gv)
        pa, ga = pv[:, 1:] - pv[:, :-1], gv[:, 1:] - gv[:, :-1]
        d_a = F.mse_loss(pa, ga)
    total = cfg['lambda_dll'] * d_ll + cfg['lambda_dvl'] * d_vl + cfg['lambda_dal'] * d_a
    return total, d_ll, d_vl, d_a


def make_optimizer(model, cfg):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if (p.ndim <= 1 or 'norm' in name.lower()) else decay).append(p)
    return torch.optim.AdamW(
        [{'params': decay, 'weight_decay': cfg['weight_decay']},
         {'params': no_decay, 'weight_decay': 0.0}], lr=cfg['lr'])


def lr_lambda_factory(cfg, steps_per_epoch):
    warmup = max(1, int(cfg['warmup_epochs'] * steps_per_epoch))
    total = max(warmup + 1, int(cfg['epochs'] * steps_per_epoch))

    def fn(step):
        if step < warmup:
            return float(step + 1) / float(warmup)
        prog = min(1.0, float(step - warmup) / float(total - warmup))
        return 0.5 * (1.0 + math.cos(math.pi * prog))

    return fn


def prune_checkpoints(keep_n):
    snaps = sorted(OUT_DIR.glob('phaseA_student_epoch_*.pth'),
                   key=lambda p: int(re.search(r'epoch_(\d+)', p.name).group(1)))
    for old in snaps[:-keep_n] if keep_n > 0 else snaps:
        try:
            old.unlink()
        except OSError:
            pass
    return [p.name for p in snaps[-keep_n:]] if keep_n > 0 else []


def train(cfg):
    model = build_model(emotion_dim=0).to(DEVICE)
    optimizer = make_optimizer(model, cfg)
    steps_per_epoch = max(1, len(train_loader))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda_factory(cfg, steps_per_epoch))
    use_amp = bool(cfg['amp']) and DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    start_epoch, global_step = 0, 0
    best_val, best_lmd, history = float('inf'), float('inf'), []
    if HISTORY_PATH.exists():
        try:
            history = json.loads(HISTORY_PATH.read_text())
        except Exception:
            history = []

    if AUTO_RESUME and CKPT_LAST.exists():
        try:
            ckpt = torch.load(CKPT_LAST, map_location=DEVICE, weights_only=False)
            model.load_state_dict_flexible(ckpt['model_state_dict'], verbose=True)
            if 'optimizer_state_dict' in ckpt:
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            if 'scheduler_state_dict' in ckpt:
                scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            start_epoch = int(ckpt.get('epoch', 0))
            global_step = int(ckpt.get('step', 0))
            # 'best_val' trong checkpoint MOI luu val LMD cua checkpoint tot nhat.
            # Checkpoint cu (truoc khi doi tieu chi) luu val loss - gia tri nay nho hon
            # val LMD rat nhieu nen neu dung lai se khien best.pth khong bao gio duoc
            # cap nhat. Phan biet bang truong 'best_metric'.
            if ckpt.get('best_metric') == 'val_lmd':
                best_lmd = float(ckpt.get('best_val', float('inf')))
                print('>> RESUME tu %s (epoch %d, step %d, best_LMD %.6f)'
                      % (CKPT_LAST.name, start_epoch, global_step, best_lmd))
            else:
                print('>> RESUME tu %s (epoch %d, step %d)' % (CKPT_LAST.name, start_epoch, global_step))
                print('   [luu y] checkpoint cu dung tieu chi val loss -> dat lai best_LMD,')
                print('           best.pth se duoc ghi lai theo tieu chi val LMD o epoch nay.')
        except Exception as exc:
            print('[canh bao] Khong resume duoc (%s) -> train moi tu dau.' % exc)

    def save(path, epoch, step, best, with_opt=False):
        model.save_checkpoint(
            path, optimizer=optimizer if with_opt else None,
            scheduler=scheduler if with_opt else None,
            epoch=epoch, step=step, best_val=best,
            extra={'history': history, 'cfg': cfg, 'device': str(DEVICE),
                   'torch': torch.__version__, 'best_metric': 'val_lmd'})

    def run_epoch(loader, train_mode, want_lmd=False, periodic_hook=None):
        nonlocal global_step
        model.train(train_mode)
        agg = np.zeros(4)
        lmd_sum, lmd_n = 0.0, 0
        nb = 0
        for mel, fl in loader:
            mel = mel.to(DEVICE, non_blocking=True)
            fl = fl.to(DEVICE, non_blocking=True)
            with torch.set_grad_enabled(train_mode):
                with torch.amp.autocast('cuda', enabled=use_amp):
                    pred = model(mel)
                    total, d_ll, d_vl, d_a = compute_loss(pred, fl, cfg)
                if train_mode:
                    optimizer.zero_grad(set_to_none=True)
                    scaler.scale(total).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    # luu dinh ky giua epoch -> Colab bi ngat cung khong mat qua nhieu
                    global_step += 1
                    if periodic_hook is not None and global_step % cfg['ckpt_every_steps'] == 0:
                        periodic_hook()
            agg += [total.item(), d_ll.item(), d_vl.item(), d_a.item()]
            nb += 1
            if want_lmd:
                with torch.no_grad():
                    d = (pred.float() - fl.float()).reshape(-1, NUM_POINTS, COORDS).norm(dim=2)
                    lmd_sum += float(d.mean()) * d.shape[0]
                    lmd_n += d.shape[0]
        return agg / max(nb, 1), (lmd_sum / max(lmd_n, 1) if want_lmd else float('nan'))


    print('\n' + '=' * 96)
    print('%-6s %-11s %-11s %-11s %-11s %-11s %-9s' %
          ('epoch', 'lr', 'loss', 'D-LL', 'D-VL', 'D-A', 'time'))
    print('=' * 96)

    def on_periodic():
        # epoch dang chay do -> ghi `epoch` (khong +1) de resume chay lai epoch nay
        save(CKPT_LAST, epoch, global_step, best_lmd, with_opt=True)
        print('%-6s [ckpt] step %d -> %s' % ('', global_step, CKPT_LAST.name))

    for epoch in range(start_epoch, cfg['epochs']):
        t0 = time.time()
        tr, _ = run_epoch(train_loader, True, periodic_hook=on_periodic)
        va, val_lmd = run_epoch(val_loader, False, want_lmd=True)
        dt = time.time() - t0
        lr_now = optimizer.param_groups[0]['lr']
        print('%-6d %-11.3e %-11.6f %-11.6f %-11.6f %-11.6f %-9s' %
              (epoch + 1, lr_now, tr[0], tr[1], tr[2], tr[3], '%.1fs' % dt))
        print('%-6s %-11s %-11.6f %-11.6f %-11.6f %-11.6f  val_LMD=%.6f' %
              ('  val', '', va[0], va[1], va[2], va[3], val_lmd))

        history.append({'epoch': epoch + 1, 'lr': lr_now, 'sec': dt,
                        'train': dict(zip(('loss', 'd_ll', 'd_vl', 'd_a'), tr.tolist())),
                        'val': dict(zip(('loss', 'd_ll', 'd_vl', 'd_a'), va.tolist())),
                        'val_lmd': val_lmd})
        HISTORY_PATH.write_text(json.dumps(history, indent=2))

        # global_step da duoc tang theo tung batch trong run_epoch
        # Chon best.pth theo val LMD (thuoc do nghiem thu) chu khong theo val loss:
        # sau diem toi uu, val loss va val LMD khong con di cung chieu.
        is_best = val_lmd < best_lmd
        if is_best:
            best_lmd = val_lmd
            save(CKPT_BEST, epoch + 1, global_step, best_lmd, with_opt=False)
        best_val = min(best_val, va[0])
        save(CKPT_LAST, epoch + 1, global_step, best_lmd, with_opt=True)
        save(OUT_DIR / ('phaseA_student_epoch_%03d.pth' % (epoch + 1)),
             epoch + 1, global_step, best_lmd, with_opt=False)
        kept = prune_checkpoints(cfg['keep_last_n'])
        print('%-6s best_LMD=%.6f%s | Drive: last.pth best.pth %s'
              % ('', best_lmd, '  <-- BEST' if is_best else '', ' '.join(kept)))

    print('=' * 96)
    print('Xong. Best val LMD = %.6f | val loss tot nhat = %.6f' % (best_lmd, best_val))

    print('Checkpoint tot nhat:', CKPT_BEST)
    return model, history


model, history = train(CFG)


## Cell 10 - Ve duong cong loss


In [ ]:
def plot_history(history, out_dir):
    if not history:
        print('Chua co history.')
        return
    ep = [h['epoch'] for h in history]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
    for ax, key, title in zip(axes,
                              ('loss', 'd_ll', 'd_vl', 'd_a'),
                              ('Total loss', 'D-LL (position)', 'D-VL (velocity)', 'D-A (acceleration)')):
        ax.plot(ep, [h['train'][key] for h in history], marker='o', ms=3, label='train')
        ax.plot(ep, [h['val'][key] for h in history], marker='s', ms=3, label='val')
        ax.set_yscale('log')
        ax.set_title(title)
        ax.set_xlabel('epoch')
        ax.grid(alpha=.3)
        ax.legend()
    fig.tight_layout()
    p = Path(out_dir) / 'phaseA_loss_curves.png'
    fig.savefig(p, dpi=130)
    plt.show()
    print('Da luu:', p)


# Neu Colab bi ngat va ban chi chay lai tu Cell 10, bien `history` khong con trong bo nho
# -> doc lai tu phaseA_history.json tren Drive (do Cell 9 ghi sau moi epoch).
if 'history' not in dir():
    _hist_path = OUT_DIR / 'phaseA_history.json'
    history = json.loads(_hist_path.read_text()) if _hist_path.exists() else []
    print('Nap history tu %s (%d epoch)' % (_hist_path.name, len(history)))

plot_history(history, OUT_DIR)


## Cell 11 - **EVALUATION CELL** - nghiem thu chat luong mo hinh

### Cell nay doc lap - chay duoc ma khong can train

Cell 11 **khong** bat buoc phai chay Cell 9 ngay truoc. No tu di tim checkpoint theo thu tu:

1. `CFG['eval_ckpt']` neu ban dien duong dan tuyet doi vao.
2. `CKPT_BEST` / `CKPT_LAST` neu Cell 9 da chay trong cung session.
3. `OUT_DIR/phaseA_student_best.pth` roi `last.pth` - truong hop mo lai notebook, chay
   Cell 1..6 roi nhay thang toi day.
4. `OUT_DIR/phaseA_student_epoch_NNN.pth` moi nhat (du phong).

Neu khong tim thay file nao, cell nem `FileNotFoundError` voi huong dan cu the.

Nap checkpoint (uu tien `best.pth`), chay suy luan tren tap `test` va tinh:

| Chi so | Y nghia | Cong thuc |
| :--- | :--- | :--- |
| **D-LL** | Do lech vi tri khong gian tinh cua 68 diem moc | $\mathrm{MSE}(\hat{y},\,y)$ |
| **D-VL** | Do lech van toc giua 2 frame lien tiep (do muot) | $\mathrm{MSE}(\Delta\hat{y},\,\Delta y)$ |
| **D-A** | Do lech gia toc (triet tieu rung/giat/flicker) | $\mathrm{MSE}(\Delta^2\hat{y},\,\Delta^2 y)$ |
| **LMD Overall** | Khoang cach Euclide thuc te tren ca 68 diem | $\frac{1}{T\cdot 68}\sum_{t,i}\lVert \hat{p}_{t,i}-p_{t,i}\rVert_2$ |
| **LMD Mouth** | Nhu tren, chi 20 diem mien mieng (index 48..67) | nhu tren, $i \in [48,67]$ |

**Don vi:** model hoc trong khong gian chuan hoa `STD_FACE_LANDMARKS`, nen LMD duoc bao cao
o dang *std-unit* (gia tri tho). Cot `LMD Baseline-mean` la moc so sanh: neu model **khong**
tot hon viec luon doan bang landmark trung binh cua tap train thi model vo dung.

### Che do suy luan - diem rat quan trong

Model **chi duoc train tren cua so `num_window_frames` (18) frame**. Suy luan tren mot doan
dai lien tuc (VD 4000 frame) la **ngoai phan phoi train**, va sai lech khong phai nho:

* Sau khi downsample $\times 16$, do dai chuoi $T' = \lfloor (T-1)/2 \rfloor + 1$.
* Voi $T \le 32$ thi $T' = 2$; voi $T = 33$ thi $T' = 3$ - co mot **buoc nhay cung**.
* Khi $T' = 2$, lop conv thu hai trong `_ConvBlock1d` (kernel 3, stride 1, pad 1) thu ve
  **1 vi tri**, khien nhanh residual gan nhu tro thanh vector hang so - dau ra mat gan het
  phu thuoc vao input.

(So lieu thuc nghiem cu duoc do tren landmark 3D CHUA can chinh nen khong con dung. Chay Cell 11 de do lai
baseline, cua so truot va chunk tren du lieu 2D da can chinh.)

Vi vay mac dinh `CFG['eval_mode'] = 'window'` (cua so truot) va khop chinh xac voi phan phoi
luc train. `stride = 8` nhanh hon `stride = 1` khoang 6 lan (hay do lai xem chat luong co giong nhau tren du lieu 2D); cac cua
so chong lan duoc trung binh cong theo tung frame.

Dat `CFG['eval_mode'] = 'chunk'` chi de **chan doan** - cell se van in so lieu chunk de doi
chieu nhung do la so lieu ngoai phan phoi, khong dung de nghiem thu.

Nghiem thu: **LMD Mouth phai tot hon baseline it nhat `CFG['eval_accept_gain']` %** (mac dinh 25, can chinh
sau lan chay dau). Khong dung nguong LMD tuyet doi vi baseline da thap hon no.


In [ ]:
# ============================ EVALUATION CELL ============================
# Cell nay DOC LAP: chi can Cell 1 -> 6 la chay duoc, khong bat buoc phai train.
CKPT_CANDIDATES = [
    OUT_DIR / 'phaseA_student_best.pth',
    OUT_DIR / 'phaseA_student_last.pth',
]


def resolve_eval_checkpoint(cfg):
    """Tim checkpoint de danh gia theo thu tu uu tien.

    1) `cfg['eval_ckpt']`  - nguoi dung chi dinh tay (duong dan tuyet doi).
    2) `CKPT_BEST` / `CKPT_LAST` - neu Cell 9 (train) da chay trong cung session.
    3) `phaseA_student_best.pth` / `last.pth` trong OUT_DIR - truong hop mo lai
       notebook, chay lai tu Cell 1 va bo qua Cell 9.
    4) `phaseA_student_epoch_NNN.pth` moi nhat trong OUT_DIR (du phong).
    """
    manual = cfg.get('eval_ckpt') or ''
    if manual:
        p = Path(os.path.expanduser(str(manual)))
        if not p.exists():
            raise FileNotFoundError("CFG['eval_ckpt'] = %s nhung file khong ton tai." % p)
        return p
    for name in ('CKPT_BEST', 'CKPT_LAST'):
        p = globals().get(name)
        if p is not None and Path(p).exists():
            return Path(p)
    for p in CKPT_CANDIDATES:
        if p.exists():
            return p

    def _last_epoch(p):
        m = re.search(r'epoch_(\d+)', p.name)
        return int(m.group(1)) if m else -1

    snaps = sorted(OUT_DIR.glob('phaseA_student_epoch_*.pth'), key=_last_epoch)
    if snaps:
        return snaps[-1]
    raise FileNotFoundError(
        'Khong tim thay checkpoint nao trong %s\n'
        '  -> Chay Cell 9 (train) truoc, hoac copy checkpoint vao thu muc tren,\n'
        "  -> hoac dien duong dan tuyet doi vao CFG['eval_ckpt'] o Cell 1." % OUT_DIR)


EVAL_CKPT   = resolve_eval_checkpoint(CFG)
REPORT_PATH = OUT_DIR / 'phaseA_evaluation.json'
print('Checkpoint:', EVAL_CKPT)
print('Kich thuoc : %.2f MB' % (EVAL_CKPT.stat().st_size / 1024 / 1024))


def _percentiles(arr, ps=(50, 90, 95)):
    """Tra ve p50/p90/p95 va max cua mot mang khoang cach."""
    out = {p: float('nan') for p in tuple(ps) + ('max',)}
    if arr.size == 0:
        return out
    for p in ps:
        out[p] = float(np.percentile(arr, p))
    out['max'] = float(arr.max())
    return out


def infer_windowed(model, mel, window=None, stride=1, batch=512):
    """Suy luan bang cua so truot dung do dai luc train, trung binh vung chong lan.

    Day la che do khop voi phan phoi ma model da hoc. Vi model rat nhe (3.1M tham so)
    nen co the gom nhieu cua so thanh mot batch de chay nhanh hon nhieu so voi vong
    lap tung cua so.
    """
    model.eval()
    window = int(window or CFG['num_window_frames'])
    T = mel.shape[0]
    if T < window:                       # video ngan hon cua so -> dem cho du
        mel = torch.nn.functional.pad(mel, (0, 0, 0, window - T))
        T = window
    starts = list(range(0, T - window + 1, int(stride)))
    if starts[-1] != T - window:
        starts.append(T - window)

    acc = np.zeros((T, LM_DIM), dtype=np.float64)
    cnt = np.zeros(T, dtype=np.float64)
    with torch.no_grad():
        for i in range(0, len(starts), batch):
            chunk_starts = starts[i:i + batch]
            x = torch.stack([mel[s:s + window] for s in chunk_starts]).to(DEVICE)
            out = model(x).float().cpu().numpy()
            for j, s in enumerate(chunk_starts):
                acc[s:s + window] += out[j]
                cnt[s:s + window] += 1.0
    cnt[cnt == 0] = 1.0
    return (acc / cnt[:, None]).astype(np.float32)


def infer_full_sequence(model, mel, chunk=None, overlap=0):
    """Chay suy luan tren ca chuoi dai, cat thanh cac chunk chong lan roi trung binh vung chong."""
    model.eval()
    T = mel.shape[0]
    chunk = int(chunk or T)
    with torch.no_grad():
        if T <= chunk:
            return model(mel.unsqueeze(0).to(DEVICE)).squeeze(0).float().cpu().numpy()
        step = max(1, chunk - int(overlap))
        acc = np.zeros((T, LM_DIM), dtype=np.float64)
        cnt = np.zeros(T, dtype=np.float64)
        for s in range(0, T, step):
            e = min(T, s + chunk)
            if e - s < 2:
                break
            out = model(mel[s:e].unsqueeze(0).to(DEVICE)).squeeze(0).float().cpu().numpy()
            acc[s:e] += out
            cnt[s:e] += 1.0
            if e >= T:
                break
        cnt[cnt == 0] = 1.0
        return (acc / cnt[:, None]).astype(np.float32)


def evaluate(ckpt_path, split='test'):
    model = StudentAudio2Landmark.from_checkpoint(ckpt_path, map_location=DEVICE)
    model = model.to(DEVICE).eval()
    ds = FullSequenceDataset(AU[split], FL[split], MEL_MEAN, MEL_STD,
                             CFG['num_window_frames'], name=split)

    # --- baseline tham chieu: luon doan bang landmark trung binh cua tap train ------
    mean_lm = np.concatenate(FL['train'], axis=0).mean(axis=0).astype(np.float32)

    tot = dict(n_frames=0, sq=0.0, sq_v=0.0, n_v=0, sq_a=0.0, n_a=0,
               lmd=0.0, lmd_mouth=0.0, n_mouth=0)
    tot_base = dict(lmd=0.0, mouth=0.0)
    tot_chunk = dict(lmd=0.0, n=0)        # chi de chan doan (xem CFG['eval_mode'])
    per_video, lmd_frames, lmd_mouth_frames = [], [], []

    for vi in range(len(ds)):
        mel, gt, _ = ds[vi]
        if CFG['eval_mode'] == 'window':
            pred = infer_windowed(model, mel, CFG['num_window_frames'],
                                  CFG['eval_window_stride'])
        else:
            pred = infer_full_sequence(model, mel, CFG['eval_chunk_frames'],
                                       CFG['eval_overlap'])
        pred = np.asarray(pred, dtype=np.float64)
        g = gt.numpy().astype(np.float64)
        T = g.shape[0]
        if pred.shape[0] != T:
            pred = pred[:T]

        d = pred - g
        tot['sq'] += float((d ** 2).sum())
        tot['n_frames'] += T

        if T > 1:
            dv = d[1:] - d[:-1]
            tot['sq_v'] += float((dv ** 2).sum())
            tot['n_v'] += dv.size
        if T > 2:
            da = d[2:] - 2 * d[1:-1] + d[:-2]
            tot['sq_a'] += float((da ** 2).sum())
            tot['n_a'] += da.size

        # LMD: khoang cach Euclide theo tung diem moc
        lmd_f = np.linalg.norm(d.reshape(T, NUM_POINTS, COORDS), axis=2)          # (T, 68)
        lmd_frames.append(lmd_f.mean(axis=1))
        lmd_mouth_frames.append(lmd_f[:, STUDENT_MOUTH_IDX].mean(axis=1))

        per_video.append({'video': vi, 'frames': T,
                          'lmd_all': float(lmd_f.mean()),
                          'lmd_mouth': float(lmd_f[:, STUDENT_MOUTH_IDX].mean())})
        tot['lmd'] += float(lmd_f.sum())
        tot['lmd_mouth'] += float(lmd_f[:, STUDENT_MOUTH_IDX].sum())
        tot['n_mouth'] += T * len(STUDENT_MOUTH_IDX)

        # baseline: luon doan landmark trung binh cua tap train
        bl = np.linalg.norm((mean_lm[None, :] - g).reshape(T, NUM_POINTS, COORDS), axis=2)
        tot_base['lmd'] += float(bl.mean()) * T
        tot_base['mouth'] += float(bl[:, STUDENT_MOUTH_IDX].mean()) * T

        # chan doan: neu dang danh gia bang cua so, do them LMD o che do chunk dai de
        # cho thay ro muc do sai lech khi suy luan ngoai phan phoi train.
        if CFG['eval_mode'] == 'window':
            pchunk = infer_full_sequence(model, mel, CFG['eval_chunk_frames'],
                                         CFG['eval_overlap'])
            pchunk = np.asarray(pchunk, dtype=np.float64)[:T]
            tot_chunk['lmd'] += float(
                np.linalg.norm((pchunk - g).reshape(T, NUM_POINTS, COORDS), axis=2).mean()) * T
            tot_chunk['n'] += T
            del pchunk

        del pred, g
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    n = tot['n_frames']
    res = OrderedDict()
    res['D-LL'] = tot['sq'] / (n * LM_DIM)
    res['D-VL'] = tot['sq_v'] / max(tot['n_v'], 1)
    res['D-A'] = tot['sq_a'] / max(tot['n_a'], 1)
    res['LMD Overall (std-unit)'] = tot['lmd'] / (n * 68)
    res['LMD Mouth (std-unit)'] = tot['lmd_mouth'] / max(tot['n_mouth'], 1)
    res['RMSE Overall (std-unit)'] = math.sqrt(res['D-LL'])
    res['LMD Baseline-mean (std-unit)'] = tot_base['lmd'] / n
    res['LMD Baseline-mouth (std-unit)'] = tot_base['mouth'] / n
    res['Gain vs baseline (std-unit)'] = 100.0 * (
        1.0 - res['LMD Overall (std-unit)'] / max(res['LMD Baseline-mean (std-unit)'], 1e-12))
    res['Gain mouth vs baseline (std-unit)'] = 100.0 * (
        1.0 - res['LMD Mouth (std-unit)'] / max(tot_base['mouth'] / n, 1e-12))

    # chan doan che do suy luan: LMD khi cat chunk dai (ngoai phan phoi train)
    res['LMD Chunk-mode (std-unit)'] = (tot_chunk['lmd'] / tot_chunk['n']
                                        if tot_chunk['n'] else float('nan'))

    stats = {'LMD Overall': _percentiles(np.concatenate(lmd_frames)),
             'LMD Mouth': _percentiles(np.concatenate(lmd_mouth_frames))}
    return model, res, stats, per_video, int(n)


model_eval, RES, STATS, PER_VIDEO, N_FRAMES = evaluate(EVAL_CKPT, 'test')

# ------------------------------------------------------------------- SUMMARY LOG
def print_table(title, headers, rows):
    rows = [[str(c) for c in r] for r in rows]
    widths = [max(len(str(h)), max([len(r[i]) for r in rows] or [0]))
              for i, h in enumerate(headers)]
    line = '+' + '+'.join('-' * (w + 2) for w in widths) + '+'
    bar = len(line)
    print()
    print('=' * bar)
    print(title.center(bar))
    print('=' * bar)
    print(line)
    print('| ' + ' | '.join(str(h).ljust(widths[i]) for i, h in enumerate(headers)) + ' |')
    print(line)
    for r in rows:
        print('| ' + ' | '.join(r[i].ljust(widths[i]) for i in range(len(headers))) + ' |')
    print(line)


print_table(
    'PHASE A - STUDENT AUDIO2LANDMARK  |  EVALUATION ON TEST SPLIT',
    ['Metric', 'Value', 'Unit', 'Meaning'],
    [
        ['D-LL  (Landmark Distance)', '%.6f' % RES['D-LL'], 'MSE',
         'Sai so vi tri tinh cua 68 diem moc'],
        ['D-VL  (Velocity Loss)', '%.6f' % RES['D-VL'], 'MSE',
         'Sai so van toc (dao ham bac 1) - do muot'],
        ['D-A   (Acceleration Loss)', '%.6f' % RES['D-A'], 'MSE',
         'Sai so gia toc (dao ham bac 2) - chong flicker'],
        ['LMD Overall', '%.6f' % RES['LMD Overall (std-unit)'], 'std-unit',
         'Euclide trung binh tren 68 diem'],
        ['LMD Mouth', '%.6f' % RES['LMD Mouth (std-unit)'], 'std-unit',
         'Euclide trung binh tren 20 diem mieng'],
        ['RMSE Overall', '%.6f' % RES['RMSE Overall (std-unit)'], 'std-unit',
         'Can bac 2 cua D-LL'],
    ])

print_table(
    'REFERENCE  |  SO SANH VOI BASELINE (landmark trung binh tap train)',
    ['Metric', 'Model', 'Baseline', 'Model tot hon'],
    [
        ['LMD Overall (std-unit)', '%.6f' % RES['LMD Overall (std-unit)'],
         '%.6f' % RES['LMD Baseline-mean (std-unit)'],
         '%.1f%%' % RES['Gain vs baseline (std-unit)']],
        ['LMD Mouth (std-unit)', '%.6f' % RES['LMD Mouth (std-unit)'],
         '%.6f' % RES['LMD Baseline-mouth (std-unit)'],
         '%.1f%%' % RES['Gain mouth vs baseline (std-unit)']],
    ])

if CFG['eval_mode'] == 'window':
    print_table(
        'DIAGNOSTIC  |  TAC DONG CUA CHE DO SUY LUAN (khong dung de nghiem thu)',
        ['Che do suy luan', 'LMD Overall', 'Ghi chu'],
        [
            ['Cua so truot %d frame (dang dung)' % CFG['num_window_frames'],
             '%.6f' % RES['LMD Overall (std-unit)'], 'khop phan phoi luc train'],
            ['Chunk dai %d frame' % CFG['eval_chunk_frames'],
             '%.6f' % RES['LMD Chunk-mode (std-unit)'],
             'ngoai phan phoi train -> chi de tham chieu'],
        ])


print_table(
    'DISTRIBUTION  |  PHAN VI LMD THEO TUNG FRAME',
    ['Series', 'p50', 'p90', 'p95', 'max'],
    [[k] + ['%.6f' % STATS[k][p] for p in (50, 90, 95)] + ['%.6f' % STATS[k]['max']]
     for k in ('LMD Overall', 'LMD Mouth')])

worst = sorted(PER_VIDEO, key=lambda r: -r['lmd_all'])[:5]
print_table(
    'TOP 5 VIDEO KEM NHAT',
    ['video idx', 'frames', 'LMD Overall', 'LMD Mouth'],
    [[r['video'], r['frames'], '%.6f' % r['lmd_all'], '%.6f' % r['lmd_mouth']] for r in worst])

print()
print('-' * 78)
print('RUN METADATA')
print('-' * 78)
print('  checkpoint      : %s' % EVAL_CKPT.name)
print('  test videos     : %d  |  test frames: %s'
      % (len(PER_VIDEO), '{:,}'.format(N_FRAMES)))
print('  window frames   : %d  |  loss weights (dll,dvl,dal) = %.2f / %.2f / %.2f'
      % (CFG['num_window_frames'], CFG['lambda_dll'], CFG['lambda_dvl'], CFG['lambda_dal']))
print('  eval mode       : %s%s'
      % (CFG['eval_mode'],
         ('  (window=%d, stride=%d)' % (CFG['num_window_frames'], CFG['eval_window_stride']))
         if CFG['eval_mode'] == 'window' else
         ('  (chunk=%d, overlap=%d)' % (CFG['eval_chunk_frames'], CFG['eval_overlap']))))
print('  parameters      : %.3f M' % (model_eval.num_parameters() / 1e6))
print('  landmark space  : 2D, can chinh tu the dau (fl_name=%s, dim=%d)' % (CFG['fl_name'], LM_DIM))
print('  device          : %s  |  torch %s' % (DEVICE, torch.__version__))
print('-' * 78)
_gm, _thr = RES['Gain mouth vs baseline (std-unit)'], CFG['eval_accept_gain']
print('  KET LUAN: %s (LMD Mouth tot hon baseline %.1f%% %s %.1f%%)'
      % ('DAT yeu cau' if _gm >= _thr else 'CHUA DAT', _gm, '>=' if _gm >= _thr else '<', _thr))
if _gm < _thr:
    print('  Goi y   : tang samples_per_epoch / chinh lambda_dvl, lambda_dal / train them')
print('-' * 78)


REPORT_PATH.write_text(json.dumps(
    {'checkpoint': str(EVAL_CKPT), 'split': 'test', 'n_frames': N_FRAMES,
     'metrics': RES, 'percentiles': STATS, 'per_video': PER_VIDEO, 'cfg': CFG},
    indent=2))
print('Da luu bao cao:', REPORT_PATH)


## Cell 12 - Xuat landmark ra `.npy` de render 2D (khong bat buoc)

Lay mot video bat ky trong tap `test`, chay model va luu ra `.npy` (T, 68, 2) - dung dinh
dang ma buoc render 2D (`facewarp` / `gen_puppet_utils.norm_anno`) mong doi.

Landmark duoc xuat bang **cung che do cua so truot** nhu Cell 11, de `.npy` nam trong phan
phoi ma model da hoc va khop voi so lieu nghiem thu.


In [ ]:
EXPORT_VIDEO = 0
EXPORT_DIR   = OUT_DIR / 'pred_landmarks'

mel = (AU['test'][EXPORT_VIDEO] - MEL_MEAN) / MEL_STD
mel = torch.from_numpy(np.ascontiguousarray(mel))
pred = infer_windowed(model_eval, mel, CFG['num_window_frames'], CFG['eval_window_stride'])

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
npy_name = re.sub(r'\.(wav|mp4)$', '', NAMES['test'][EXPORT_VIDEO]) or ('video_%d' % EXPORT_VIDEO)
npy_path = EXPORT_DIR / ('%s_pred.npy' % npy_name)
np.save(npy_path, pred.reshape(-1, NUM_POINTS, COORDS).astype(np.float32))
# Delta so voi landmark trung binh tap train: cong vao landmark cua mat dich de retarget
delta_path = EXPORT_DIR / ('%s_delta.npy' % npy_name)
np.save(delta_path, (pred - FL_MEAN[None, :]).reshape(-1, NUM_POINTS, COORDS).astype(np.float32))
np.save(EXPORT_DIR / 'fl_mean_2d.npy', FL_MEAN.reshape(NUM_POINTS, COORDS))
gt_path = EXPORT_DIR / ('%s_gt.npy' % npy_name)
np.save(gt_path, FL['test'][EXPORT_VIDEO].reshape(-1, NUM_POINTS, COORDS).astype(np.float32))
print('video   :', NAMES['test'][EXPORT_VIDEO])
print('pred    :', npy_path, pred.reshape(-1, NUM_POINTS, COORDS).shape)
print('gt      :', gt_path)
print()
print('Dinh dang .npy (T, 68, 2) nay dung truc tiep cho buoc render 2D:')
print('  1. ap dung post-process goc cua MakeItTalk (savgol + close-mouth) neu can')
print('  2. facewarp.exe / facewarp 2D triangle morphing -> video')


## Ghi chu Phase 2 (Emotion Vector alpha) - khong phai cell thuc thi

Phase 1 ket thuc khi `phaseA_student_best.pth` dat chat luong nghiem thu. Khong can
train lai tu dau o Phase 2:

```python
# --- Phase 2: tai checkpoint Phase 1 va gan nhanh emotion ------------------------
model = StudentAudio2Landmark.from_checkpoint(
    '/content/drive/MyDrive/DAT301m/Data_student/student_a2l/phaseA/phaseA_student_best.pth',
    emotion_dim=8,               # so chieu cua emotion vector alpha
    emotion_fuse='concat',
    map_location=DEVICE,
).to(DEVICE)

# output ngay sau khi nang cap GIONG HET Phase 1 (emotion_gate=1, cot moi cua fc[0]=0)
# tu day tro di chi can fine-tune: toan bo conv giu nguyen shape.

# (tuy chon) dong bang backbone de chi hoc nhanh emotion truoc:
model.set_encoder_trainable(False)   # ... train vai epoch ...
model.set_encoder_trainable(True)    # ... roi mo khoa fine-tune toan bo
```

### Trang thai du lieu Phase 2

Du lieu MEAD dang duoc pipeline Stage-B trich xuat (`Data_emotion_mead_task_b`).
Khi co `manifest.json` + `landmark_package.zip` cho tung speaker thi moi chay Phase 2.
Notebook nay **khong** dung `emb.pickle` (speaker embedding 256D) - do la huong mo rong
khac, co the cam vao bang cach doi `emotion_dim=256` va truyen embedding lam `emotion`.
